In [45]:
from  langgraph.graph import START,StateGraph,END
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
import os
from typing import TypedDict

In [46]:
load_dotenv()

True

In [47]:
AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT_EUS2')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_APIKEY_EUS2')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
LLM_DEPLOYMENT_NAME = os.getenv('LLM_DEPLOYMENT_NAME', os.getenv('MODEL_NAME'))


azure_chat_client = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=LLM_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.2,
)

In [48]:
#crate a state

class LLMState(TypedDict):
    question: str
    answer: str

In [49]:
from langchain_core.messages import HumanMessage
def llm_qa(state: LLMState) -> LLMState:
    question = state['question']
    prompt = f"Answer the question: {question}"
    # AzureChatOpenAI expects a list of message objects or a string
    response = azure_chat_client.invoke(prompt)
    answer = response.content
    return LLMState(question=question, answer=answer)

In [50]:
graph = StateGraph(LLMState)


#add nodes
graph.add_node('llm_qa', llm_qa)
#add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)   

workflow = graph.compile()


In [51]:
initial_state = LLMState(question="how far is moon from earth?", answer="")
final_state = workflow.invoke(initial_state)

print(final_state)


{'question': 'how far is moon from earth?', 'answer': 'The Moon is about **384,400 km** (**238,855 miles**) from Earth on average.\n\nBecause its orbit is elliptical, the distance varies:\n- **Closest (perigee):** about **363,300 km** (**225,700 miles**)\n- **Farthest (apogee):** about **405,500 km** (**251,900 miles**)'}


In [52]:
azure_chat_client.invoke("where is india?")

AIMessage(content='India is in **South Asia**.\n\nIt is:\n- **south of China**\n- **east of Pakistan**\n- **west of Bangladesh and Myanmar**\n- bordered by the **Indian Ocean** to the south\n\nIts capital is **New Delhi**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 10, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05-short-api-ev3', 'system_fingerprint': None, 'id': 'chatcmpl-DQEFffzrjHySkUYPxV5CKXfDLI52U', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'